In [4]:
from http.server import HTTPServer, BaseHTTPRequestHandler

class Upload(BaseHTTPRequestHandler):
  def do_PUT(self):
      length = int(self.headers['Content-Length'])
      with open('/workspace/amd-crusher/l2_bundle.tar.gz', 'wb') as f:
          remaining = length
          while remaining > 0:
              chunk = self.rfile.read(min(65536, remaining))
              f.write(chunk)
              remaining -= len(chunk)
      self.send_response(200)
      self.end_headers()
      self.wfile.write(b'OK')
      print(f'Received {length} bytes')

HTTPServer(('0.0.0.0', 8080), Upload).serve_forever()

127.0.0.1 - - [19/Mar/2026 19:35:25] "PUT /l2_bundle.tar.gz HTTP/1.1" 200 -


Received 1473859348 bytes


KeyboardInterrupt: 

In [5]:
!cd /workspace/amd-crusher && tar xzf l2_bundle.tar.gz
!wc -l /workspace/amd-crusher/data/v10_train_manifest_rx.csv /workspace/amd-crusher/data/val_manifest_rx.csv

  32065 /workspace/amd-crusher/data/v10_train_manifest_rx.csv
   2643 /workspace/amd-crusher/data/val_manifest_rx.csv
  34708 total


In [10]:
!pip install torchao --upgrade 2>&1 | tail -3


In [6]:
import csv, os

for f in ['data/v10_train_manifest_rx.csv', 'data/val_manifest_rx.csv']:
  path = f'/workspace/amd-crusher/{f}'
  rows = []
  with open(path) as fh:
      reader = csv.DictReader(fh)
      fields = reader.fieldnames
      for row in reader:
          row['wav_path'] = row['wav_path'].replace('/root/workspace/amd-crusher/', '/workspace/amd-crusher/')
          rows.append(row)
  with open(path, 'w', newline='') as fh:
      writer = csv.DictWriter(fh, fieldnames=fields)
      writer.writeheader()
      writer.writerows(rows)
  exists = sum(1 for r in rows if os.path.isfile(r['wav_path']))


In [13]:
import subprocess, os

seeds = [137, 42, 99, 73]
gpus = [1, 6, 0, 2]  # GPUs with most free memory

procs = []
for seed, gpu in zip(seeds, gpus):
  cmd = (
      f"cd /workspace/amd-crusher && CUDA_VISIBLE_DEVICES={gpu} "
      f"python3 scripts/finetune_wav2vec2_v9.py "
      f"--train-csv data/v10_train_manifest_rx.csv "
      f"--val-csv data/val_manifest_rx.csv "
      f"--num-classes 3 --curriculum-epochs 2 "
      f"--min-clip-seconds 0.5 --truncation-prob 1.0 "
      f"--augment --augment-humans-only "
      f"--unfreeze-layers 8 --lr 1e-5 --batch-size 64 --grad-accum-steps 1 "
      f"--max-epochs 25 --patience 6 "
      f"--focal-gamma 2.0 --seed {seed} "
      f"--callguard-weight 8.0 "
      f"--output-dir output/l2_v11_seed{seed}"
  )
  log = open(f"/workspace/train_seed{seed}.log", "w")
  p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)
  procs.append((seed, gpu, p))
  print(f"Started seed={seed} on GPU {gpu} (PID {p.pid})")

print(f"\n4 training jobs launched! Monitor with:")
for seed, gpu, p in procs:
  print(f"  !tail -5 /workspace/train_seed{seed}.log")



Started seed=137 on GPU 1 (PID 2937068)
Started seed=42 on GPU 6 (PID 2937069)
Started seed=99 on GPU 0 (PID 2937070)
Started seed=73 on GPU 2 (PID 2937072)

4 training jobs launched! Monitor with:
  !tail -5 /workspace/train_seed137.log
  !tail -5 /workspace/train_seed42.log
  !tail -5 /workspace/train_seed99.log
  !tail -5 /workspace/train_seed73.log


In [15]:
import time

while True:
  time.sleep(60)
  alive = []
  for seed, gpu, p in procs:
      rc = p.poll()
      if rc is None:
          alive.append(seed)
          # Show latest epoch
          !tail -1 /workspace/train_seed{seed}.log | grep -E "^\s+[0-9]|Epoch|Phase"
      else:
          print(f"  Seed {seed}: FINISHED (exit {rc})")
  if not alive:
      print("ALL DONE!")
      break
  print(f"  Still running: seeds {alive}\n")

  [Phase 1] Training with 42672 balanced samples (long audio >3000ms)
  [Phase 1] Training with 42672 balanced samples (long audio >3000ms)
  [Phase 1] Training with 42672 balanced samples (long audio >3000ms)
  Seed 73: FINISHED (exit 1)
  Still running: seeds [137, 42, 99]

  [Phase 1] Training with 42672 balanced samples (long audio >3000ms)
  Seed 73: FINISHED (exit 1)
  Still running: seeds [137, 42, 99]

  Seed 73: FINISHED (exit 1)
  Still running: seeds [137, 42, 99]

  Seed 73: FINISHED (exit 1)
  Still running: seeds [137, 42, 99]

  Seed 73: FINISHED (exit 1)
  Still running: seeds [137, 42, 99]

  [Phase 2] Expanding to 57663 balanced samples (+medium audio 1500-3000ms)
  [Phase 2] Expanding to 57663 balanced samples (+medium audio 1500-3000ms)
  Seed 73: FINISHED (exit 1)
  Still running: seeds [137, 42, 99]

  [Phase 2] Expanding to 57663 balanced samples (+medium audio 1500-3000ms)
  [Phase 2] Expanding to 57663 balanced samples (+medium audio 1500-3000ms)
  [Phase 2] Ex

In [18]:
import json, glob

for d in sorted(glob.glob("/workspace/amd-crusher/output/l2_v11_seed*/best_metrics.json")):
  m = json.load(open(d))
  seed = d.split("seed")[1].split("/")[0]
  print(f"Seed {seed}: min(H,M,CG)={m['min_all_recall']:.1f}% | H={m['human_recall']:.1f}% M={m['machine_recall']:.1f}% CG={m['callguard_recall']:.1f}%")



Seed 137: min(H,M,CG)=40.2% | H=40.2% M=40.2% CG=94.6%
Seed 42: min(H,M,CG)=30.2% | H=30.2% M=74.8% CG=91.8%
Seed 99: min(H,M,CG)=38.5% | H=38.5% M=56.7% CG=94.6%


In [19]:
seeds = [137, 42]                                                                                                                                                                     
gpus = [1, 6]                                                                                                                                                                         
                                                                                                                                                                                    
for seed, gpu in zip(seeds, gpus):                                                                                                                                                    
  cmd = (                                                                                                                                                                           
      f"cd /workspace/amd-crusher && CUDA_VISIBLE_DEVICES={gpu} "                                                                                                                   
      f"python3 scripts/finetune_wav2vec2_v9.py "                                                                                                                                   
      f"--train-csv data/v10_train_manifest_rx.csv "                                                                                                                                
      f"--val-csv data/val_manifest_rx.csv "                                                                                                                                        
      f"--num-classes 3 --curriculum-epochs 2 "                                                                                                                                     
      f"--min-clip-seconds 0.5 --truncation-prob 1.0 "                                                                                                                              
      f"--augment --augment-humans-only "                                                                                                                                           
      f"--unfreeze-layers 6 --lr 2e-5 --batch-size 64 --grad-accum-steps 1 "                                                                                                        
      f"--max-epochs 25 --patience 8 "                                                                                                                                              
      f"--focal-gamma 1.0 --seed {seed} "                                                                                                                                           
      f"--callguard-weight 3.0 "                                                                                                                                                    
      f"--output-dir output/l2_v12_seed{seed}"                                                                                                                                      
  )                                                                                                                                                                                 
  log = open(f"/workspace/train_v12_seed{seed}.log", "w")                                                                                                                           
  p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)                                                                                                       
  print(f"Started seed={seed} on GPU {gpu} (PID {p.pid})")                                                                                                                          
                                                                                                                                                                                    
print("Monitor: !tail -5 /workspace/train_v12_seed137.log")                                                                                                                           
      

Started seed=137 on GPU 1 (PID 2967482)
Started seed=42 on GPU 6 (PID 2967483)
Monitor: !tail -5 /workspace/train_v12_seed137.log


In [21]:
import subprocess

configs = [
  (42, 1),   # seed 42 on GPU 1
  (137, 6),  # seed 137 on GPU 6
]

for seed, gpu in configs:
  cmd = (
      f"cd /workspace/amd-crusher && CUDA_VISIBLE_DEVICES={gpu} "
      f"python3 scripts/finetune_wav2vec2_v9.py "
      f"--train-csv data/v10_train_manifest_rx.csv "
      f"--val-csv data/val_manifest_rx.csv "
      f"--num-classes 3 "
      f"--manual-class-weights 1.2,1.0,1.5 "
      f"--focal-gamma 1.0 "
      f"--curriculum-epochs 0 "
      f"--augment "
      f"--unfreeze-layers 6 "
      f"--lr 3e-5 "
      f"--batch-size 32 --grad-accum-steps 2 "
      f"--max-epochs 40 --patience 12 "
      f"--min-clip-seconds 0.5 --truncation-prob 1.0 "
      f"--seed {seed} "
      f"--output-dir output/l2_v13_seed{seed}"
  )
  log = open(f"/workspace/train_v13_seed{seed}.log", "w")
  p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)
  print(f"Seed {seed} on GPU {gpu} — PID {p.pid}")



Seed 42 on GPU 1 — PID 2976255
Seed 137 on GPU 6 — PID 2976256
